In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
# All librariesd that were there in the program
import kagglehub
import os
import glob
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
from tqdm import tqdm
import matplotlib.pyplot as plt

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, image_paths, masks, transform=None, target_transform=None):
        self.image_paths = image_paths
        self.masks = masks
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]  # Get image path
        mask_path = self.masks[idx]  # Get corresponding mask path

        # Load image and mask
        image = Image.open(image_path).convert('RGB')
        mask = Image.open(mask_path)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        if isinstance(mask, torch.Tensor):
            mask = remap_mask(mask)

        return image, mask  # Return processed image and its mask

# Setup paths
root_dir = os.path.join(path, "dataset")
print(root_dir)
images = glob.glob(f"{root_dir}/images/*.jpg")
print(f"Number of images: {len(images)}")
masks = glob.glob(f"{root_dir}/masks/*.png")
print(f"Number of masks: {len(masks)}")

# Define custom transform to subtract 1 from mask
class SubtractOne:
    def __call__(self, img):
        return img - 1

# Define transforms for images
img_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define transforms for masks
target_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.PILToTensor(),
    SubtractOne()
])

# Split dataset into 80% train, 20% test
train_data, test_data, train_mask, test_mask = train_test_split(
    images, masks, test_size=0.2, random_state=42, shuffle=True
)

# Load train and test datasets
train_dataset = CustomDataset(
    train_data, train_mask, transform=img_transforms, target_transform=target_transforms
)
test_dataset = CustomDataset(
    test_data, test_mask, transform=img_transforms, target_transform=target_transforms
)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train Dataset: {len(train_dataset)} images")
print(f"Test Dataset: {len(test_dataset)} images")

In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# Define U-Net Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = smp.Unet(
    encoder_name="efficientnet-b0",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Number of classes (0-7)
).to(device)


In [ ]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader, desc="Training"):
        images, masks = images.to(device), masks.to(device)

        # Squeeze the mask from [N, 1, H, W] to [N, H, W] and convert to long
        masks = masks.squeeze(1).long()

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc="Validation"):
            images, masks = images.to(device), masks.to(device)

            # Squeeze the mask from [N, 1, H, W] to [N, H, W] and convert to long
            masks = masks.squeeze(1).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)



In [ ]:
criterion = nn.CrossEntropyLoss()

# Define optimizer
optimizer = optim.AdamW(model.parameters(), lr=0.001)

num_epochs = 10
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")


In [ ]:
# TO DO
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(True)
plt.show()

# Function to visualize predictions
def visualize_predictions(model, dataset, device, num_samples=3):
    model.eval()
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))

    indices = torch.randint(0, len(dataset), (num_samples,))

    for i, idx in enumerate(indices):
        image, true_mask = dataset[idx]
        image = image.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(image)
            pred_mask = torch.argmax(output, dim=1).squeeze(0).cpu()

        # Denormalize image for visualization
        image_np = image.squeeze(0).cpu().permute(1, 2, 0).numpy()
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3)
        image_np = image_np * std.numpy() + mean.numpy()
        image_np = np.clip(image_np, 0, 1)

        true_mask_np = true_mask.squeeze(0).cpu().numpy()
        pred_mask_np = pred_mask.numpy()

        axes[i, 0].imshow(image_np)
        axes[i, 0].set_title('Input Image')
        axes[i, 0].axis('off')

        axes[i, 1].imshow(true_mask_np, cmap='tab20')
        axes[i, 1].set_title('True Mask')
        axes[i, 1].axis('off')

        axes[i, 2].imshow(pred_mask_np, cmap='tab20')
        axes[i, 2].set_title('Predicted Mask')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

# Visualize some predictions
visualize_predictions(model, test_dataset, device, num_samples=3)
